In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import cobra
from cobra.core import configuration
from PolyRound.api import PolyRoundApi
from PolyRound.settings import PolyRoundSettings
import hopsy

In [3]:
configuration.Configuration.solver = 'glpk'

In [4]:
model = cobra.io.read_sbml_model('./../autopacmen_output/Mitocore_aligned_to_Human1/MitoCore_aligned_to_Human1_calibrated_MA.xml')

In [5]:
model.optimize()

,fluxes,reduced_costs
EX_2hb_e,0.000000e+00,0.0
EX_ac_e,-1.151314e-01,0.0
EX_acac_e,0.000000e+00,0.0
EX_akg_e,0.000000e+00,0.0
EX_ala_B_e,0.000000e+00,0.0
...,...,...
ENZYME_DELIVERY_ENSG00000058063,0.000000e+00,-0.0
ENZYME_DELIVERY_ENSG00000124406,0.000000e+00,-0.0
ENZYME_DELIVERY_ENSG00000085231,4.790755e-08,0.0
ENZYME_DELIVERY_ENSG00000102743,0.000000e+00,-0.0


# 1) Sanity check: Model has a solution with lower bound for objective function

In [6]:
# for reaction in model.reactions:
#     if reaction.upper_bound == 0.0 and reaction.lower_bound == 0.0:
#         model.remove_reactions(reaction)

# model.reactions.OF_ATP_MitoCore.lower_bound = 1.707042254 * 0.95
model.optimize()

,fluxes,reduced_costs
EX_2hb_e,0.000000e+00,0.0
EX_ac_e,-1.151314e-01,0.0
EX_acac_e,0.000000e+00,0.0
EX_akg_e,0.000000e+00,0.0
EX_ala_B_e,0.000000e+00,0.0
...,...,...
ENZYME_DELIVERY_ENSG00000058063,0.000000e+00,-0.0
ENZYME_DELIVERY_ENSG00000124406,0.000000e+00,-0.0
ENZYME_DELIVERY_ENSG00000085231,4.790755e-08,0.0
ENZYME_DELIVERY_ENSG00000102743,0.000000e+00,-0.0


In [7]:
cobra.io.write_sbml_model(model, './../flux_sampling/chrr_reounding_test_h1_ma.xml')

# 2) Reproduction of the Error 

In [10]:
# reinitialize hopsies settings for polyround to use glpk solver
settings = PolyRoundSettings(backend='glpk')
hopsy.LP.settings = settings
hopsy.LP.settings.__dict__

{'backend': 'glpk',
 'hp_flags': {'FeasibilityTol': 1e-09},
 'thresh': 1e-07,
 'verbose': False,
 'sgp': False,
 'reduce': True,
 'regularize': False,
 'check_lps': False,
 'simplify_only': False,
 'presolve': False,
 'numerics_threshold': 1e-12,
 'accepted_tol_violation': 100.0}

In [11]:
polytope = PolyRoundApi.sbml_to_polytope('./../flux_sampling/chrr_reounding_test_h1_ma.xml')
problem = hopsy.Problem(polytope.A, polytope.b) # set the polytype, i.e., the inequality constraints from the constraint based model
problem = hopsy.add_equality_constraints(problem=problem, A_eq=polytope.S, b_eq=polytope.h)
start = time.perf_counter()
problem = hopsy.round(problem)
print(f"Computing rounding transformation for {model_name} took {time.perf_counter()-start} seconds")

1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
0.0
0.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
0.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
0.0
0.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
0.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
1000.0
100

ValueError: Adding these equality constraints makes the problem infeasible! Check the problem and/or the LP().settings

# 3) Sanity check with MitoMammal model

In [ ]:
polytope = PolyRoundApi.sbml_to_polytope('./../flux_sampling/Mitocore_MitoMammal_sampling_model.xml')
problem = hopsy.Problem(polytope.A, polytope.b) # set the polytype, i.e., the inequality constraints from the constraint based model
problem = hopsy.add_equality_constraints(problem=problem, A_eq=polytope.S, b_eq=polytope.h)
start = time.perf_counter()
problem = hopsy.round(problem)
print(f"Computing rounding transformation for {model_name} took {time.perf_counter()-start} seconds")

GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

Original model can be rounded, looks like its an issue with ec models

In [ ]:
polytope = PolyRoundApi.sbml_to_polytope('./../flux_sampling/Mitocore_Original_plt_sampling_model.xml')
problem = hopsy.Problem(polytope.A, polytope.b) # set the polytype, i.e., the inequality constraints from the constraint based model
problem = hopsy.add_equality_constraints(problem=problem, A_eq=polytope.S, b_eq=polytope.h)
start = time.perf_counter()
problem = hopsy.round(problem)
print(f"Computing rounding transformation for {model_name} took {time.perf_counter()-start} seconds")

NameError: name 'model_name' is not defined

# 4) Trying to change LP settings

In [ ]:
# reinitialize hopsies settings for polyround to use glpk solver
settings = PolyRoundSettings(backend='glpk', hp_flags = {"FeasibilityTol": 1e-6}, thresh = 1e-12, verbose=True)
hopsy.LP.settings = settings
hopsy.LP.settings.__dict__

{'backend': 'glpk',
 'hp_flags': {'FeasibilityTol': 1e-06},
 'thresh': 1e-12,
 'verbose': True,
 'sgp': False,
 'reduce': True,
 'regularize': False,
 'check_lps': False,
 'simplify_only': False,
 'presolve': False,
 'numerics_threshold': 1e-12,
 'accepted_tol_violation': 100.0}

In [ ]:
polytope = PolyRoundApi.sbml_to_polytope('./../flux_sampling/Mitocore_MitoMammal_sampling_model.xml')
problem = hopsy.Problem(polytope.A, polytope.b) # set the polytype, i.e., the inequality constraints from the constraint based model
problem = hopsy.add_equality_constraints(problem=problem, A_eq=polytope.S, b_eq=polytope.h)
print('1')
start = time.perf_counter()
problem = hopsy.round(problem)
print(f"Computing rounding transformation for {model_name} took {time.perf_counter()-start} seconds")


 Investigating constraint number: 0


 Investigating constraint number: 50


 Investigating constraint number: 100


 Investigating constraint number: 150


 Investigating constraint number: 200


 Investigating constraint number: 250


 Investigating constraint number: 300


 Investigating constraint number: 350


 Investigating constraint number: 400


 Investigating constraint number: 450


 Investigating constraint number: 500


 Investigating constraint number: 550


 Investigating constraint number: 600


 Investigating constraint number: 650


 Investigating constraint number: 700


 Investigating constraint number: 750


 Investigating constraint number: 800


 Investigating constraint number: 850


 Investigating constraint number: 900


 Investigating constraint number: 950


 Investigating constraint number: 1000


 Investigating constraint number: 1050


 Investigating constraint number: 1100


 Investigating constraint number: 1150


 Investigating constraint number: 1200